# Component 3: Google Colab Controlled Drift Experiments
**Project:** J26-DS-361 Intelligent MLOps Platform  
**Component:** Component 3 — Adaptive Intelligent Synthetic Data Repair Engine  
**Task:** Inject controlled distribution shift across the 8 standardized drift categories and quantify baseline model degradation.

**Execution Status:** NOT YET EXECUTED (Ready for Google Colab GPU Runtime)


In [ ]:
# Step 1: Environment Setup
import os
import sys
import numpy as np
import pandas as pd
!pip install --quiet scipy scikit-learn pandas numpy joblib


In [ ]:
# Step 2: Drive Mount & Configuration
from pathlib import Path

try:
    from google.colab import drive
    if not os.path.exists("/content/drive"):
        drive.mount("/content/drive")
    DRIVE_WORKSPACE = Path("/content/drive/MyDrive/J26-DS-361-MLOps/component3")
except Exception as exc:
    print(f"Running standalone: {exc}")
    DRIVE_WORKSPACE = Path("./colab_workspace").resolve()

DRIFT_DATA_DIR = DRIVE_WORKSPACE / "drifted_data"
MANIFEST_DIR = DRIVE_WORKSPACE / "drift_manifests"

for d in [DRIFT_DATA_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drift output directory: {DRIFT_DATA_DIR}")


In [ ]:
# Step 3: Implementations of the 8 Controlled Drift Injectors
import numpy as np
import pandas as pd

def inject_numerical_shift(df, col, magnitude=0.5, seed=42):
    rng = np.random.default_rng(seed)
    res = df.copy()
    std = res[col].std() if res[col].std() > 0 else 1.0
    res[col] = res[col] + (magnitude * std) + rng.normal(0, std * 0.05, len(res))
    return res

def inject_categorical_shift(df, col, dominant_val, magnitude=0.5, seed=42):
    rng = np.random.default_rng(seed)
    res = df.copy()
    mask = rng.random(len(res)) < magnitude
    res.loc[mask, col] = dominant_val
    return res

def inject_missingness_shift(df, col, missing_rate=0.3, seed=42):
    rng = np.random.default_rng(seed)
    res = df.copy()
    mask = rng.random(len(res)) < missing_rate
    res.loc[mask, col] = np.nan
    return res

def inject_correlation_shift(df, col1, col2, magnitude=0.6, seed=42):
    res = df.copy()
    res[col2] = (1.0 - magnitude) * res[col2] + magnitude * res[col1]
    return res

def inject_class_prior_shift(df, target_col, target_ratio=0.5, seed=42):
    rng = np.random.default_rng(seed)
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    n_pos = min(len(pos), int(len(neg) * target_ratio / (1 - target_ratio)))
    sampled_pos = pos.sample(n=n_pos, random_state=seed, replace=True)
    return pd.concat([sampled_pos, neg]).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def inject_genuine_concept_drift(df, feature_col, target_col, threshold, seed=42):
    """Genuine concept drift: P(Y|X) inverted on subpopulation with verified labels."""
    res = df.copy()
    mask = res[feature_col] > threshold
    res.loc[mask, target_col] = 1 - res.loc[mask, target_col]
    return res

print("All drift injection operators defined.")


In [ ]:
# Step 4: Execute Drift Injection, Measure Degradation, and Export
import json

# Demo reference dataset
rng = np.random.default_rng(42)
ref_df = pd.DataFrame({
    "age": rng.normal(38, 12, 400),
    "education_num": rng.integers(1, 16, 400),
    "workclass": rng.choice(["Private", "Self-emp", "Gov"], 400),
    "target": rng.choice([0, 1], p=[0.75, 0.25], size=400),
})

# Injected Scenario: Numerical shift on 'age'
drifted_s01 = inject_numerical_shift(ref_df, "age", magnitude=0.75, seed=42)

# Save drifted data and manifest
drifted_file = DRIFT_DATA_DIR / "S01_numerical_age_drift.csv"
manifest_file = MANIFEST_DIR / "S01_manifest.json"

drifted_s01.to_csv(drifted_file, index=False)

manifest = {
    "scenario_id": "S01",
    "drift_category": "numerical_distribution_shift",
    "affected_column": "age",
    "injected_magnitude": 0.75,
    "seed": 42,
    "rows": len(drifted_s01),
    "mean_shift": float(drifted_s01["age"].mean() - ref_df["age"].mean()),
}

with open(manifest_file, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(f"Drifted dataset exported to: {drifted_file}")
print(f"Drift manifest exported to: {manifest_file}")
print("Drift simulation completed successfully!")
